In [0]:
print("E-Commerce Data Engineering Project Started!")

In [0]:
d = [
    (1, "Chandana", 500),
    (2, "Rahul", 750),
    (3, "Priya", 1200)
]
c = ["customer_id", "customer_name", "total_spent"]
df = spark.createDataFrame(data=d, schema=c)
df.write.mode("overwrite").saveAsTable("customer")


In [0]:
display(df)

In [0]:
spark.sql("SELECT * FROM customer").show()

In [0]:
%sql
SELECT * FROM customer

In [0]:
%sql
select "hello mani" as message


In [0]:
%ls

> **dbutils.widgets.dropdown** - used for creating a new dropdown in the UI

> **dbutils.widgets.text** - used for creating a new text input in the UI

> **dbutils.widgets.get** - used for retrieving the value of a widget

> **%run** - used for running another notebook in the current notebook context

> **%run ./includes/utilities**
 DBTITLE 1,Create a new widget
 
> **dbutils.widgets.text("message", "")**
 DBTITLE 1,Create a new widget ''' 

In [0]:

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
dbutils.widgets.get("env")

In [0]:
from pyspark.sql.functions import lit
dbutils.widgets.text("message", "")
message = dbutils.widgets.get("message")
print(message)
df = spark.sql("SELECT * FROM customer")
display(df)
df = df.withColumn("message", lit(message))
df.write.mode("overwrite").saveAsTable("_sqldf")

# Spark Write Modes Demo

This notebook demonstrates the **4 main write modes** in Apache Spark:
1. **append** - Adds new records to existing table
2. **overwrite** - Replaces all existing data
3. **ignore** - Does nothing if table exists
4. **errorifexists** (default) - Throws error if table exists

---

In [0]:
# Create initial sample data for demo
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

print("📊 Creating INITIAL dataset...\n")

initial_data = [
    Row(order_id=1, product="Laptop", quantity=2, price=1200.00, status="Delivered"),
    Row(order_id=2, product="Mouse", quantity=5, price=25.00, status="Shipped"),
    Row(order_id=3, product="Keyboard", quantity=3, price=75.00, status="Pending")
]

df_initial = spark.createDataFrame(initial_data)
df_initial = df_initial.withColumn("created_at", current_timestamp())

print("Initial Dataset:")
df_initial.show(truncate=False)
print(f"Total Records: {df_initial.count()}")

In [0]:
# DEMO 1: Using mode="overwrite" to create the table
print("\n" + "="*60)
print("📝 DEMO 1: mode='overwrite' - Creating table for the first time")
print("="*60 + "\n")

table_name = "demo_orders"

# Write initial data with overwrite mode
df_initial.write.mode("overwrite").saveAsTable(table_name)

print(f"✅ Table '{table_name}' created successfully!\n")
print("Table Contents:")
spark.sql(f"SELECT * FROM {table_name}").show(truncate=False)
print(f"📌 Total Records: {spark.sql(f'SELECT COUNT(*) as count FROM {table_name}').collect()[0]['count']}")

In [0]:
# DEMO 2: Using mode="append" to add new records
print("\n" + "="*60)
print("📝 DEMO 2: mode='append' - Adding NEW records to existing table")
print("="*60 + "\n")

# Create NEW data to append
new_data = [
    Row(order_id=4, product="Monitor", quantity=1, price=350.00, status="Processing"),
    Row(order_id=5, product="Webcam", quantity=2, price=120.00, status="Delivered")
]

df_new = spark.createDataFrame(new_data)
df_new = df_new.withColumn("created_at", current_timestamp())

print("New records to APPEND:")
df_new.show(truncate=False )


# Append the new data
df_new.write.mode("append").saveAsTable(table_name)

print(f"\n✅ New records APPENDED to '{table_name}'!\n")
print("Updated Table Contents:")
spark.sql(f"SELECT * FROM {table_name} ORDER BY order_id").show(truncate=False)
print(f"📌 Total Records NOW: {spark.sql(f'SELECT COUNT(*) as count FROM {table_name}').collect()[0]['count']} (was 3, now 5)")

In [0]:
# DEMO 3: Using mode="overwrite" to REPLACE all existing data
print("\n" + "="*60)
print("📝 DEMO 3: mode='overwrite' - REPLACING all existing data")
print("="*60 + "\n")

print("⚠️  Current table has 5 records. Watch what happens...\n")

# Create completely NEW dataset
replacement_data = [
    Row(order_id=101, product="Tablet", quantity=10, price=500.00, status="New"),
    Row(order_id=102, product="Charger", quantity=20, price=30.00, status="New")
]

df_replacement = spark.createDataFrame(replacement_data)
df_replacement = df_replacement.withColumn("created_at", current_timestamp())

print("Replacement data (only 2 records):")
df_replacement.show(truncate=False)

# Overwrite the entire table
df_replacement.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✅ Table '{table_name}' COMPLETELY REPLACED!\n")
print("Table Contents NOW (OLD data is GONE):")
spark.sql(f"SELECT * FROM {table_name} ORDER BY order_id").show(truncate=False)
print(f"📌 Total Records NOW: {spark.sql(f'SELECT COUNT(*) as count FROM {table_name}').collect()[0]['count']} (was 5, now only 2!)")

In [0]:
# DEMO 4: Using mode="ignore" - Does nothing if table exists
print("\n" + "="*60)
print("📝 DEMO 4: mode='ignore' - Attempting to write with ignore mode")
print("="*60 + "\n")

print("Current table contents:")
spark.sql(f"SELECT * FROM {table_name}").show(truncate=False)
print(f"Current count: {spark.sql(f'SELECT COUNT(*) as count FROM {table_name}').collect()[0]['count']}\n")

# Try to write new data with ignore mode
ignore_data = [
    Row(order_id=999, product="HeadPhones", quantity=100, price=200.00, status="Ignored")
]

df_ignore = spark.createDataFrame(ignore_data)
df_ignore = df_ignore.withColumn("created_at", current_timestamp())

print("Attempting to write this data with mode='ignore':")
df_ignore.show(truncate=False)

df_ignore.write.mode("ignore").saveAsTable(table_name)

print("\n✅ Write operation completed with 'ignore' mode\n")
print("Table Contents (UNCHANGED because table already exists):")
spark.sql(f"SELECT * FROM {table_name}").show(truncate=False)
print(f"📌 Total Records: {spark.sql(f'SELECT COUNT(*) as count FROM {table_name}').collect()[0]['count']} (still 2 - no change!)")

In [0]:
# DEMO 5: Using mode="errorifexists" or "error" (default) - Throws error if table exists
print("\n" + "="*60)
print("📝 DEMO 5: mode='errorifexists' - Will throw error if table exists")
print("="*60 + "\n")

error_data = [
    Row(order_id=888, product="Speakers", quantity=50, price=150.00, status="Error Mode")
]

df_error = spark.createDataFrame(error_data)
df_error = df_error.withColumn("created_at", current_timestamp())

print("Attempting to write with mode='errorifexists':")
df_error.show(truncate=False)

try:
    df_error.write.mode("errorifexists").saveAsTable(table_name)
    print("✅ Write successful")
except Exception as e:
    print("\n❌ ERROR CAUGHT (as expected):")
    print(f"Error Type: {type(e).__name__}")
    print(f"Error Message: {str(e)[:200]}...")
    print("\n💡 This is the DEFAULT behavior - prevents accidental overwrites!")

## 📊 Summary of Write Modes

| Mode | Behavior | Use Case |
|------|----------|----------|
| **append** | Adds new records to existing table | Incremental data loads, adding daily/hourly data |
| **overwrite** | Replaces ALL existing data | Full refresh, complete data replacement |
| **ignore** | Does nothing if table exists | Safe writes, idempotent operations |
| **errorifexists** | Throws error if table exists (DEFAULT) | Prevents accidental overwrites |

---

### 🎯 Best Practices:
- Use **append** for incremental ETL pipelines
- Use **overwrite** only when you intentionally want to replace all data
- Use **ignore** for safe, repeatable scripts
- **errorifexists** is the default - protects against mistakes!

### 🔗 Bonus: Partition Overwrite
For partitioned tables, you can use:
```python
df.write.mode("overwrite").insertInto("table_name")  # Overwrites matching partitions only
```

In [0]:
# Cleanup - Uncomment to drop the demo table
print("\n🧹 Cleanup Section\n")
print("To drop the demo table, uncomment the line below:")
print(f"# spark.sql('DROP TABLE IF EXISTS {table_name}')")

# Uncomment to clean up:
# spark.sql(f"DROP TABLE IF EXISTS {table_name}")
# print(f"✅ Table '{table_name}' dropped successfully!")

---
# ⚠️ Corrupt Data Handling Modes

These modes control how Spark **reads** data with corrupt/malformed records:
1. **PERMISSIVE** (default) - Keeps corrupt records in `_corrupt_record` column
2. **DROPMALFORMED** - Silently drops corrupt rows
3. **FAILFAST** - Throws error on first corrupt record

**Key Difference:** Write modes control *how to save* data, while these modes control *how to read* corrupt data.

---

In [0]:
# Create sample CSV data with some corrupt records
import os

print("📁 Creating sample CSV file with CORRUPT records...\n")

# Sample data with intentional corruption
csv_content = """order_id,product,quantity,price,status
1,Laptop,2,1200.50,Delivered
2,Mouse,5,25.00,Shipped
CORRUPT_ROW_MISSING_COLUMNS,Keyboard,75.00
4,Monitor,1,350.00,Processing
5,Webcam,INVALID_NUMBER,120.00,Delivered
6,Headphones,3,80.00,Pending
ANOTHER_BAD_ROW
8,Tablet,2,450.00,Shipped"""

# Write to volume
volume_path = "/Volumes/hive_metastore/default/pulsecart/corrupt_orders.csv"

# Use dbutils to write the file
dbutils.fs.put(volume_path, csv_content, overwrite=True)

print(f"✅ Created file at: {volume_path}")
print(f"\n📋 File Contents (including CORRUPT rows):\n")
print(csv_content)
print("\n⚠️  Notice: Rows 3, 5, and 7 are intentionally malformed!")

In [0]:
# MODE 1: PERMISSIVE - Keeps corrupt records in _corrupt_record column
print("\n" + "="*70)
print("📝 MODE 1: mode='PERMISSIVE' (Default) - KEEPS corrupt records")
print("="*70 + "\n")

print("🔍 Reading CSV with mode='PERMISSIVE'...\n")

df_permissive = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load(volume_path)

print("✅ Read completed! Showing ALL rows (including corrupt ones):\n")
df_permissive.show(truncate=False)

print(f"\n📊 Total rows read: {df_permissive.count()}")
print("\n💡 Notice: Corrupt rows have data in '_corrupt_record' column!")
print("   Other columns are NULL for corrupt rows.\n")

# Show only corrupt records
print("🔴 Filtering ONLY corrupt records:")
df_permissive.filter("_corrupt_record IS NOT NULL").show(truncate=False)

In [0]:
# MODE 2: DROPMALFORMED - Silently drops corrupt records
print("\n" + "="*70)
print("📝 MODE 2: mode='DROPMALFORMED' - DROPS corrupt records silently")
print("="*70 + "\n")

print("🔍 Reading CSV with mode='DROPMALFORMED'...\n")

df_dropmalformed = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .load(volume_path)

print("✅ Read completed! Showing ONLY valid rows:\n")
df_dropmalformed.show(truncate=False)

print(f"\n📊 Total rows read: {df_dropmalformed.count()}")
print("\n💡 Notice: Corrupt rows are GONE! Only clean data remains.")
print("   No '_corrupt_record' column exists.")
print("   ⚠️  Dropped rows: 3 corrupt records were silently removed!")

In [0]:
# MODE 3: FAILFAST - Throws error immediately on corrupt record
print("\n" + "="*70)
print("📝 MODE 3: mode='FAILFAST' - FAILS immediately on corrupt data")
print("="*70 + "\n")

print("🔍 Attempting to read CSV with mode='FAILFAST'...\n")

try:
    df_failfast = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("mode", "FAILFAST") \
        .load(volume_path)
    
    # Try to trigger the read (Spark is lazy)
    print("Attempting to show data...")
    df_failfast.show()
    print("✅ Read successful (no corrupt data found)")
    
except Exception as e:
    print("❌ ERROR CAUGHT (as expected when corrupt data exists):\n")
    print(f"Error Type: {type(e).__name__}")
    print(f"\nError Message (first 300 chars):")
    print(str(e)[:300] + "...")
    print("\n💡 FAILFAST mode stops processing immediately when it finds corrupt data!")
    print("   This is useful when data quality is critical and you want to be alerted.")

## 📊 Comparison: Write Modes vs Corrupt Data Handling Modes

### 🔵 Write Modes (Saving Data)
Control **how to write** DataFrames to tables/files:

| Mode | Purpose | When It Acts |
|------|---------|-------------|
| **append** | Add new records | During write operation |
| **overwrite** | Replace all data | During write operation |
| **ignore** | Skip if exists | During write operation |
| **errorifexists** | Fail if exists | During write operation |

**Usage:** `df.write.mode("append").saveAsTable("table")`

---

### 🔴 Corrupt Data Handling Modes (Reading Data)
Control **how to handle corrupt records** when reading:

| Mode | Behavior | When It Acts |
|------|----------|-------------|
| **PERMISSIVE** | Keep corrupt in `_corrupt_record` column | During read operation |
| **DROPMALFORMED** | Silently drop corrupt rows | During read operation |
| **FAILFAST** | Throw error immediately | During read operation |

**Usage:** `spark.read.option("mode", "PERMISSIVE").csv("file.csv")`

---

### 🎯 When to Use Each:

**Write Modes:**
- Use when controlling how to **save** clean data
- Applied to DataFrames you're writing

**Corrupt Data Modes:**
- Use when **reading** potentially dirty/malformed source files
- Applied during data ingestion/loading
- Common in ETL pipelines processing external data

In [0]:
# REAL-WORLD EXAMPLE: Complete ETL Pipeline
print("\n" + "="*70)
print("🏭 REAL-WORLD EXAMPLE: ETL Pipeline with Both Mode Types")
print("="*70 + "\n")

print("Step 1: READ with DROPMALFORMED (corrupt data handling)")
print("        → Drop bad records during ingestion\n")

df_clean = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .load(volume_path)

print(f"✅ Loaded {df_clean.count()} clean records\n")

print("Step 2: Transform the data")
from pyspark.sql.functions import col, when, current_timestamp

df_transformed = df_clean \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("processed_at", current_timestamp())

print("✅ Data transformed\n")

print("Step 3: WRITE with OVERWRITE mode (write mode)")
print("        → Replace entire target table\n")

target_table = "clean_orders"
df_transformed.write.mode("overwrite").saveAsTable(target_table)

print(f"✅ Data saved to '{target_table}' table\n")
print("Final Result:")
spark.sql(f"SELECT * FROM {target_table}").show(truncate=False)

print("\n🎉 Complete! Used BOTH mode types in one pipeline:")
print("   1. DROPMALFORMED (read mode) - cleaned corrupt records")
print("   2. OVERWRITE (write mode) - saved clean data to table")

In [0]:
# Cleanup - Remove demo files and tables
print("\n🧹 Cleanup Section for Corrupt Data Demo\n")

print("To clean up, uncomment the lines below:")
print(f"# dbutils.fs.rm('{volume_path}')")
print(f"# spark.sql('DROP TABLE IF EXISTS clean_orders')")

# Uncomment to clean up:
# dbutils.fs.rm(volume_path)
# spark.sql('DROP TABLE IF EXISTS clean_orders')
# print("✅ Cleanup completed!")